# Inference Model

## Goal
Create a final CSV with three columns:
- **ID**: copied from the input CSV
- **Prompt**: copied from the input CSV
- **Answer**: generated by the model as one continuous text as output

## Approach
- a local instruction-tuned language model: **Qwen/Qwen2.5-1.5B-Instruct**
- 8-bit quantization to reduce GPU memory usage (and therefore runtime)
- batch processing to improve throughput (maximizing GPU memory usage at runtime)
- resume logic so interrupted runs can continue
- autosave after every batch for robustness

## Why left padding?
For decoder-only models like Qwen, left padding is important during generation.
Right padding can lead to poor generation quality in batched inference.

In [1]:
# Import libraries

import os
import math
import time
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig # BitsAndBytes for Quantization

C:\Users\lauri\PycharmProjects\DS4_llm_inference_sft_rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configuration

Here static params are defined that will be used throughout the notebook

In [ ]:
# Model name, CSV input and output path and name
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
CSV_PATH = "./data/dataset_clean_worktime_version.csv"
OUT_PATH = "./outputs/inference_results_local_8bit_inf.csv"

# Starting batch size for parallel processing of several inputs
BATCH_SIZE = 4
# Amount of tokens to produce
MAX_NEW_TOKENS = 120
# Amount of tokens to take as maximum input
MAX_INPUT_LENGTH = 768

# System prompt on how the model acts (i.e. role)
SYSTEM_PROMPT = ("You are a careful Austrian tax law assistant. "
    "Answer briefly, factually, and clearly in one plain-text paragraph. "
    "Do not invent legal references.")

## 2. GPU Configuration

Here the GPU is being checked to select it (ran via CUDA; CUDA is used to access the GPU for such tasks).
Otherwise, questions could not be answered locally with CPU due to extreme runtime.
Locally GPU used in this case:

**NVIDA RTX 5070 12 GB VRAM**

In [ ]:
# Raise error in case of missing CUDA (i.e. GPU not available or not recognized)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA not available!")

# Enable TensorFloat 32 acceleration for improved speed for NVIDA GPUs
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

## 3. Load Tokenizer

Use AutoTokenizer for tokenization.
Use 8-bit quantization for less memory usage.
AutoModelForCausalLM for text generation.

In [ ]:
quant_config = BitsAndBytesConfig(load_in_8bit=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# Padding required for Qwen models during batched generation
tokenizer.padding_side = "left"

# Set padding token in case of missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load quantized model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto")

# Set model to evaluation mode
model.eval()

# Debugging prints
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Model device:", next(model.parameters()).device)

## 4. Load Input CSV

Loading the CSV with ID and prompt as input.

Split only at first comma (maxsplit=1)

In [ ]:
# List for CSV rows
rows = []

# Open CSV file
with open(CSV_PATH, "r", encoding="utf-8-sig") as f:
    next(f, None) # Skip header row

    # Start in second line, strip it, skip missing lines and skip lines with missing separator
    for line_number, line in enumerate(f, start=2):
        line = line.strip()
        if not line:
            continue
        if "," not in line:
            print(f"Row {line_number} skipped with no comma")
            continue
        # Set variables to ID and prompt for each row
        id_, prompt = line.split(",", 1)
        rows.append({
            "ID": id_.strip(),
            "Prompt": prompt.strip()})

# Convert to pandas DF
input_df = pd.DataFrame(rows)
input_df.head()

## 5. Continuation Logic

Here we continue if the run was interrupted and continue working on the interrupted file

In [ ]:
if os.path.exists(OUT_PATH):
    existing_df = pd.read_csv(OUT_PATH, encoding="utf-8-sig")
    done_ids = set(existing_df["ID"].astype(str))
    print(f"Resume: {len(done_ids)} entries completed")
else:
    existing_df = pd.DataFrame(columns=["ID", "Prompt", "Answer"])
    done_ids = set()

work_df = input_df[~input_df["ID"].astype(str).isin(done_ids)].copy()
work_df.reset_index(drop=True, inplace=True)

print(f"Continuing with: {len(work_df)} / {len(input_df)}")

if len(work_df) == 0:
    print("All IDs finished already")
    raise SystemExit

## 6. Build Chat Input Text

Simple function for building the proper input format for the model

In [ ]:
def build_chat_text(prompt: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt}]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True)

## 7. Batch Processing Logic

Following is done:
1. Take several prompts
2. Convert to chat-formatted text
3. Tokenize together with padding
4. Generate answers
5. Decode generated output
6. Store results
7. Save progress after each batch

Saving is done in case of the program crashing or any other issue arises (CUDA out of memory, PC restart, etc.)

In [ ]:
# List for new results
new_results = []
# Number of batches to work on
num_batches = math.ceil(len(work_df) / BATCH_SIZE)
# Logging for total runtime
start_time = time.time()


for batch_idx in range(num_batches):
    batch_start = batch_idx * BATCH_SIZE # define starting IDX of certain batch for prompts
    batch_end = min(batch_start + BATCH_SIZE, len(work_df)) # define end IDX of certain batch for prompts
    batch_df = work_df.iloc[batch_start:batch_end]

    # Extract the batch data
    prompts = batch_df["Prompt"].tolist() # store all prompts
    ids = batch_df["ID"].tolist() # store all ids
    chat_texts = [build_chat_text(p) for p in prompts] # create the entire input text for the LLM

    # Tokenize the full batch
    inputs = tokenizer(
        chat_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LENGTH)

    # Move all tensors to the model device (i.e. GPU)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Generate outputs (without gradient tracking)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id)

    # Input sequences in batch padded to same length
    input_token_len = inputs["input_ids"].shape[1]

    # List for storing results
    batch_results = []

    # Go through all batch IDs
    for i in range(len(ids)):
        # Keep only newly generated tokens
        generated_tokens = outputs[i][input_token_len:]
        # Decode answer text
        answer = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
        # Convert answer into one continuous line (helps with output in CSV later)
        answer = " ".join(answer.split())

        batch_results.append({
            "ID": ids[i],
            "Prompt": prompts[i],
            "Answer": answer})

    # Add batch results to total new results
    new_results.extend(batch_results)

    # Save progress after each batch (concatenate existing df and new df)
    temp_df = pd.concat(
        [existing_df, pd.DataFrame(new_results, columns=["ID", "Prompt", "Answer"])],
        ignore_index=True)
    temp_df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

    # Logging for estimated total runtime
    processed = batch_end # last processed batch
    elapsed = time.time() - start_time # elapsed time
    per_item = elapsed / processed # duration per batch
    remaining = len(work_df) - processed # remaining batches
    eta_min = (per_item * remaining) / 60 # duration per batch times remaining batches in minutes

    print(
        f"Batch {batch_idx + 1}/{num_batches} finished | "
        f"{processed}/{len(work_df)} prompts | "
        f"ETA approx. {eta_min:.1f} min | "
        f"Elapsed time: {elapsed:.2f} min")

## 8. Final Output

Ouput returns a CSV with ID, Prompt and Answer (of model)

In [ ]:
print(f"All batches processed!")